# Silver to Gold ETL Demo Notebook

This notebook demonstrates how to build ML feature tables from the Silver layer.

**Output tables:**
- `training_features` - ML training dataset: features from orders [N-4, N-3, N-2], label from order N-1
- `serving_features` - ML serving dataset: features from orders [N-3, N-2, N-1], predict order N

## 1. Configure Env

In [ ]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2
%additional_python_modules 
%extra_jars 
%%configure
{
    "--datalake-formats": "iceberg",
    "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO --conf spark.sql.catalog.glue_catalog.warehouse=s3://instacart-aws-data-ml-eng-project/iceberg-warehouse/"
}

In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Spark version:", spark.version)
print("Initialization complete!")

In [ ]:
BUCKET_NAME = "instacart-aws-data-ml-eng-project"
CATALOG_NAME = "glue_catalog"
SILVER_DB = "silver"
GOLD_DB = "gold"
MIN_ORDERS_REQUIRED = 5

## 2. Create Gold Database

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{GOLD_DB} LOCATION 's3://{BUCKET_NAME}/{GOLD_DB}/'")
spark.sql(f"SHOW DATABASES IN {CATALOG_NAME}").show()

## 3. Training Features

Feature window: orders [N-4, N-3, N-2] — Label: order N-1

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG_NAME}.{GOLD_DB}.training_features
USING iceberg
TBLPROPERTIES ('format-version' = '2')
AS
WITH user_order_bounds AS (
    SELECT
        user_id,
        MAX(order_number) AS max_order_number
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders
    GROUP BY user_id
    HAVING MAX(order_number) >= {MIN_ORDERS_REQUIRED}
),
global_product_features AS (
    SELECT
        fop.product_id,
        COUNT(fop.order_id)                AS item_total_sales,
        AVG(CAST(fop.reordered AS DOUBLE)) AS item_reorder_rate
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    GROUP BY fop.product_id
),
train_feature_orders AS (
    SELECT
        fo.order_id,
        fo.user_id,
        fo.order_number,
        fo.days_since_prior_order,
        ub.max_order_number
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders fo
    INNER JOIN user_order_bounds ub ON fo.user_id = ub.user_id
    WHERE fo.order_number BETWEEN (ub.max_order_number - 4) AND (ub.max_order_number - 2)
),
train_label_orders AS (
    SELECT fo.order_id, fo.user_id
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders fo
    INNER JOIN user_order_bounds ub ON fo.user_id = ub.user_id
    WHERE fo.order_number = (ub.max_order_number - 1)
),
train_candidates AS (
    SELECT DISTINCT fop.user_id, fop.product_id
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    INNER JOIN train_feature_orders tfo ON fop.order_id = tfo.order_id
),
train_positive_labels AS (
    SELECT DISTINCT fop.user_id, fop.product_id, 1 AS label
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    INNER JOIN train_label_orders tlo ON fop.order_id = tlo.order_id
),
train_up_features AS (
    SELECT
        fop.user_id,
        fop.product_id,
        COUNT(fop.order_id)                                   AS user_product_buy_count,
        MAX(tfo.order_number)                                 AS user_product_last_order_in_window,
        AVG(fop.add_to_cart_order)                            AS user_product_avg_add_to_cart,
        MAX(tfo.max_order_number) - 2 - MAX(tfo.order_number) AS user_product_orders_since_last_buy
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    INNER JOIN train_feature_orders tfo ON fop.order_id = tfo.order_id
    GROUP BY fop.user_id, fop.product_id
),
train_user_features AS (
    SELECT
        user_id,
        COUNT(DISTINCT order_id)    AS user_window_orders,
        AVG(days_since_prior_order) AS user_avg_days_between,
        MAX(max_order_number)       AS user_total_orders
    FROM train_feature_orders
    GROUP BY user_id
)
SELECT
    tc.user_id,
    tc.product_id,
    up.user_product_buy_count,
    up.user_product_last_order_in_window,
    up.user_product_avg_add_to_cart,
    up.user_product_orders_since_last_buy,
    uf.user_window_orders,
    uf.user_avg_days_between,
    uf.user_total_orders,
    CASE
        WHEN uf.user_window_orders > 0 THEN up.user_product_buy_count / uf.user_window_orders
        ELSE 0
    END AS user_product_window_reorder_ratio,
    gpf.item_total_sales,
    gpf.item_reorder_rate,
    CAST(COALESCE(tpl.label, 0) AS INT) AS label,
    current_timestamp() AS audit_transform_time
FROM train_candidates tc
INNER JOIN train_up_features up
    ON tc.user_id = up.user_id AND tc.product_id = up.product_id
INNER JOIN train_user_features uf
    ON tc.user_id = uf.user_id
LEFT JOIN global_product_features gpf
    ON tc.product_id = gpf.product_id
LEFT JOIN train_positive_labels tpl
    ON tc.user_id = tpl.user_id AND tc.product_id = tpl.product_id
""")
print(f"✅ Created {CATALOG_NAME}.{GOLD_DB}.training_features")

## 4. Serving Features

Feature window: orders [N-3, N-2, N-1] — Predict: order N (no label)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG_NAME}.{GOLD_DB}.serving_features
USING iceberg
TBLPROPERTIES ('format-version' = '2')
AS
WITH user_order_bounds AS (
    SELECT
        user_id,
        MAX(order_number) AS max_order_number
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders
    GROUP BY user_id
    HAVING MAX(order_number) >= {MIN_ORDERS_REQUIRED}
),
global_product_features AS (
    SELECT
        fop.product_id,
        COUNT(fop.order_id)                AS item_total_sales,
        AVG(CAST(fop.reordered AS DOUBLE)) AS item_reorder_rate
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    GROUP BY fop.product_id
),
serve_feature_orders AS (
    SELECT
        fo.order_id,
        fo.user_id,
        fo.order_number,
        fo.days_since_prior_order,
        ub.max_order_number
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders fo
    INNER JOIN user_order_bounds ub ON fo.user_id = ub.user_id
    WHERE fo.order_number BETWEEN (ub.max_order_number - 3) AND (ub.max_order_number - 1)
),
serve_candidates AS (
    SELECT DISTINCT fop.user_id, fop.product_id
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    INNER JOIN serve_feature_orders sfo ON fop.order_id = sfo.order_id
),
serve_up_features AS (
    SELECT
        fop.user_id,
        fop.product_id,
        COUNT(fop.order_id)                                   AS user_product_buy_count,
        MAX(sfo.order_number)                                 AS user_product_last_order_in_window,
        AVG(fop.add_to_cart_order)                            AS user_product_avg_add_to_cart,
        MAX(sfo.max_order_number) - 1 - MAX(sfo.order_number) AS user_product_orders_since_last_buy
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products fop
    INNER JOIN serve_feature_orders sfo ON fop.order_id = sfo.order_id
    GROUP BY fop.user_id, fop.product_id
),
serve_user_features AS (
    SELECT
        user_id,
        COUNT(DISTINCT order_id)    AS user_window_orders,
        AVG(days_since_prior_order) AS user_avg_days_between,
        MAX(max_order_number)       AS user_total_orders
    FROM serve_feature_orders
    GROUP BY user_id
)
SELECT
    sc.user_id,
    sc.product_id,
    up.user_product_buy_count,
    up.user_product_last_order_in_window,
    up.user_product_avg_add_to_cart,
    up.user_product_orders_since_last_buy,
    uf.user_window_orders,
    uf.user_avg_days_between,
    uf.user_total_orders,
    CASE
        WHEN uf.user_window_orders > 0 THEN up.user_product_buy_count / uf.user_window_orders
        ELSE 0
    END AS user_product_window_reorder_ratio,
    gpf.item_total_sales,
    gpf.item_reorder_rate,
    current_timestamp() AS audit_transform_time
FROM serve_candidates sc
INNER JOIN serve_up_features up
    ON sc.user_id = up.user_id AND sc.product_id = up.product_id
INNER JOIN serve_user_features uf
    ON sc.user_id = uf.user_id
LEFT JOIN global_product_features gpf
    ON sc.product_id = gpf.product_id
""")
print(f"✅ Created {CATALOG_NAME}.{GOLD_DB}.serving_features")

## 5. Verify Gold Tables

In [ ]:
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{GOLD_DB}").show()

spark.table(f"{CATALOG_NAME}.{GOLD_DB}.training_features").show(5)
spark.sql(f"SELECT COUNT(*) as total FROM {CATALOG_NAME}.{GOLD_DB}.training_features").show()

spark.sql(f"SELECT label, COUNT(*) as cnt FROM {CATALOG_NAME}.{GOLD_DB}.training_features GROUP BY label ORDER BY label").show()

In [ ]:
spark.table(f"{CATALOG_NAME}.{GOLD_DB}.serving_features").show(5)
spark.sql(f"SELECT COUNT(*) as total FROM {CATALOG_NAME}.{GOLD_DB}.serving_features").show()

## 6. Sample Queries

In [ ]:
# Query 1: Label distribution in training set
spark.sql(f"""
    SELECT
        label,
        COUNT(*) AS cnt,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {CATALOG_NAME}.{GOLD_DB}.training_features
    GROUP BY label
    ORDER BY label
""").show()

In [ ]:
# Query 2: Top products by item_reorder_rate in serving features
spark.sql(f"""
    SELECT product_id, item_total_sales, ROUND(item_reorder_rate, 3) AS item_reorder_rate
    FROM {CATALOG_NAME}.{GOLD_DB}.serving_features
    ORDER BY item_reorder_rate DESC
    LIMIT 10
""").show()

In [ ]:
# Query 3: Distribution of user_product_window_reorder_ratio for positive vs negative labels
spark.sql(f"""
    SELECT
        label,
        ROUND(AVG(user_product_window_reorder_ratio), 3) AS avg_window_reorder_ratio,
        ROUND(AVG(user_product_buy_count), 2)            AS avg_buy_count,
        ROUND(AVG(user_product_orders_since_last_buy), 2) AS avg_orders_since_last_buy
    FROM {CATALOG_NAME}.{GOLD_DB}.training_features
    GROUP BY label
    ORDER BY label
""").show()

## 7. Clean Up (Optional)

In [ ]:
# Uncomment to drop all gold tables
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{GOLD_DB}.training_features")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{GOLD_DB}.serving_features")
# spark.sql(f"DROP DATABASE IF EXISTS {CATALOG_NAME}.{GOLD_DB}")